In [1]:
import sys

sys.path.append("/kaggle/input/models/jxyle1/revist-tiny-stories-gpt2-configs/pytorch/default/1")
sys.path.append("/kaggle/input/models/jxyle1/basline-gpt2/pytorch/default/1")

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken

from GPT import GPT2
from configs import GPTConfig

from huggingface_hub import HfApi


In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="karpathy/tinystories-gpt4-clean",
    repo_type="dataset",
    local_dir="./KarpathySories"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/kaggle/working/KarpathySories'

In [5]:
import pandas as pd
df = pd.read_parquet("./KarpathySories/tinystories_gpt4_clean.parquet")

with open("./KarpathySories/tinystories.txt", "w", encoding="utf-8") as f:
    for story in df['text']:
        f.write(story)
        f.write("\n\n")

print("Finshed!")


Finshed!


In [7]:
total_batch_size = 65536 # 0.5M in number of tokens

batch_size = 1
block_size = 2048
grad_accum_steps = total_batch_size // (batch_size * block_size)

max_iters = 36622
eval_interval = 200
checkpoint_iters = 1000

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [8]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

target_tokens = 300_000_000
tokens = []
encode = lambda s: enc.encode(s)
decode = lambda l: enc.decode(l)
vocab_size = enc.n_vocab

with open("./KarpathySories/tinystories.txt", "r", encoding="utf-8") as f:
    while len(tokens) < target_tokens:
        chunk = f.read(5_000_000)
        if not chunk:
            break

        tokens.extend(enc.encode(chunk))

tokens = tokens[:target_tokens]

data = torch.tensor(tokens,dtype=torch.long)

print(f"Token length : {len(tokens)}")


Token length : 300000000


In [9]:
from huggingface_hub import login

login()

api = HfApi()
def save_checkpoint(iter,model,optimizer,loss):
    checkpoint = {
        "epoch":iter,
        "model" : model.state_dict(),
        "optimizer":optimizer.state_dict(),
        "loss": loss,
    }

    filename = f"E004-latest_checkpoint.pt"

    torch.save(checkpoint,filename)

    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=f"checkpoints/{filename}",
        repo_id="MLGOATKOJI/Refresh-TinyStories",
        repo_type="model",
    )

    print(f"Uploaded {filename}")

In [10]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)

    return x.to(device), y.to(device)

In [11]:
model = GPT2(GPTConfig())
model = model.to(device)
model = torch.compile(model)
max_lr = 3e-4
min_lr = max_lr * 0.1
warmup_steps = int(max_iters * 0.02)
max_steps = max_iters

import math
def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it+1)/warmup_steps

    if it > max_steps:
        return min_lr

    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0<= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)
optimizer = torch.optim.AdamW(model.parameters(),lr=max_lr)

    

In [12]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_interval)
        for k in range(eval_interval):
            X, Y = get_batch(split)
            X,Y = X.to(device), Y.to(device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [13]:
#-------------------------------CHECKPOINT LOADING-------------------

from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="MLGOATKOJI/Refresh-TinyStories",
    filename="checkpoints/E004-latest_checkpoint.pt"
)
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])

start_iter = checkpoint["epoch"] + 1

checkpoints/E004-latest_checkpoint.pt: reconstructing file:   0%|          |  0.00B /  369MB            

checkpoints/E004-latest_checkpoint.pt: downloading bytes:           |  0.00B            

In [14]:
scaler = torch.amp.GradScaler("cuda")
for iter in range(start_iter,max_iters):


    if iter % eval_interval == 0 or iter == max_iters -1:#
        losses = estimate_loss()
        print(f"Step: {iter}: Train Loss: {losses['train']:.4f}: Val Loss: {losses['val']:.4f}: Learning rate: {get_lr(iter):.4e}")

    if iter % checkpoint_iters == 0 or iter == max_iters - 1:
        save_checkpoint(iter,model,optimizer,losses)

    

    optimizer.zero_grad()
    loss_accum = 0.0
    for micro_step in range(grad_accum_steps):
        x, y = get_batch("train")
        x,y = x.to(device), y.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits, loss = model(x, y)

        loss = loss / grad_accum_steps
        loss_accum += loss.detach()
        scaler.scale(loss).backward()

        
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = get_lr(iter)

    
    scaler.step(optimizer)
    scaler.update()

    
        

W1101 08:47:33.672000 49 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Step: 35200: Train Loss: 1.4114: Val Loss: 1.4242: Learning rate: 3.1044e-05
Step: 35400: Train Loss: 1.4178: Val Loss: 1.4140: Learning rate: 3.0772e-05
Step: 35600: Train Loss: 1.3971: Val Loss: 1.4117: Learning rate: 3.0540e-05
Step: 35800: Train Loss: 1.4083: Val Loss: 1.4336: Learning rate: 3.0349e-05
Step: 36000: Train Loss: 1.4200: Val Loss: 1.4204: Learning rate: 3.0200e-05


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded E004-latest_checkpoint.pt
Step: 36200: Train Loss: 1.4061: Val Loss: 1.4087: Learning rate: 3.0092e-05
Step: 36400: Train Loss: 1.4109: Val Loss: 1.4183: Learning rate: 3.0025e-05
Step: 36600: Train Loss: 1.4140: Val Loss: 1.4079: Learning rate: 3.0000e-05
Step: 36621: Train Loss: 1.4051: Val Loss: 1.4152: Learning rate: 3.0000e-05


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded E004-latest_checkpoint.pt


In [ ]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)


print(f"Trainable: {trainable_params:,}")


Trainable: 30,750,720
